In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Iskay Quantum Optimizer - A Qiskit Function by Kipu Quantum

> **Note:** * Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (via IBM Quantum Platform API) Plan. Nasa preview release status ito at maaaring magbago.

## Pangkalahatang-ideya
Gamit ang Iskay Quantum Optimizer ng Kipu Quantum, maaari kang harapin ang mga kumplikadong optimization problem gamit ang mga IBM&reg; quantum computer. Ginagamit ng solver na ito ang makabagong [bf-DCQO algorithm](https://doi.org/10.48550/arXiv.2409.04477) ng Kipu na nangangailangan lang ng objective function bilang input para awtomatikong maihatid ang mga solusyon. Kayang hawakan nito ang mga optimization problem na may hanggang 156 qubit, na nagbibigay-daan sa paggamit ng lahat ng qubit ng mga IBM quantum device. Gumagamit ang Optimizer ng 1-to-1 na mapping sa pagitan ng mga classical variable at qubit, kaya kayang harapin ang mga optimization problem na may hanggang 156 binary variable.

Ang Optimizer ay nagbibigay-kakayahang lutasin ang mga unconstrained binary optimization problem. Bukod sa karaniwang ginagamit na QUBO (Quadratic Unconstrained Binary Optimization) na formulation, sinusuportahan din nito ang mga higher-order (HUBO) optimization problem. Gumagamit ang solver ng non-variational quantum algorithm na nagpapatakbo ng karamihan ng computation sa mga quantum device.

Nagbibigay ang sumusunod ng mas detalyadong impormasyon tungkol sa algorithm na ginamit at isang maikling gabay sa kung paano gamitin ang function, kasama na ang mga resulta ng benchmarking sa iba't ibang instance ng problema sa iba't ibang laki at kumplikasyon.

## Paglalarawan
Ang Optimizer ay isang ready-to-use na implementasyon ng mga makabagong quantum optimization algorithm. Niresolba nito ang mga optimization problem sa pamamagitan ng pagpapatakbo ng highly-compressed na quantum circuit sa quantum hardware. Nakamit ang compression na ito sa pamamagitan ng pagpapakilala ng mga counterdiabatic term sa pinagbabatayan na time evolution ng quantum system. Nagsasagawa ang algorithm ng ilang iteration ng hardware run para makuha ang mga huling solusyon at pinagsama ito sa post-processing. Ang mga hakbang na ito ay walang putol na isinama sa workflow ng Optimizer at awtomatikong naisasagawa.

### Paano gumagana ang Quantum Optimizer?
Inililahad ng seksyong ito ang mga pangunahing kaalaman ng implemented na bf-DCQO algorithm. Makikita rin ang isang panimula sa algorithm sa [Qiskit YouTube channel.](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s)

Batay ang algorithm sa time evolution ng isang quantum system na binabago sa paglipas ng panahon, kung saan ang solusyon ng problema ay naka-encode sa ground state ng quantum system sa katapusan ng evolution. Ayon sa [adiabatic theorem](https://en.wikipedia.org/wiki/Adiabatic_theorem), ang evolution na ito ay kailangang mabagal para matiyak na nananatili ang system sa ground state nito. Ang pag-digitize ng evolution na ito ang batayan ng digitized quantum adiabatic computation (DQA) at ng kilalang QAOA algorithm. Gayunpaman, ang kinakailangang mabagal na evolution ay hindi praktikal para sa mga lumalagong laki ng problema dahil nagreresulta ito sa pagtaas ng circuit depth. Sa pamamagitan ng paggamit ng mga counterdiabatic protocol, maaaring sugpuin ang mga hindi gustong excitation na nagaganap sa maikling evolution time habang nananatili sa ground state. Dito, ang pag-digitize ng mas maikling evolution time na ito ay nagreresulta sa mga quantum circuit na mas maikli ang depth at may mas kaunting entangling gate.

Karaniwang gumagamit ang mga circuit ng bf-DCQO algorithm ng hanggang sampung beses na mas kaunting entangling gate kaysa DQA, at tatlo hanggang apat na beses na mas kaunti kaysa sa mga standard na QAOA implementation. Dahil sa mas kaunting bilang ng gate, mas kaunting error ang nagaganap sa panahon ng circuit execution sa hardware. Kaya naman, hindi nangangailangan ang optimizer ng paggamit ng mga technique tulad ng error suppression o error mitigation. Ang pag-implement ng mga ito sa mga susunod na bersyon ay maaaring lalo pang mapabuti ang kalidad ng solusyon.

Bagaman gumagamit ng iteration ang bf-DCQO algorithm, ito ay non-variational. Pagkatapos ng bawat iteration ng algorithm, sinusukat ang distribusyon ng mga estado. Ang nakuhang distribusyon ay ginagamit para kalkulahin ang tinatawag na bias-field. Nagbibigay-daan ang bias-field na simulan ang susunod na iteration mula sa energy state na malapit sa dating nahanap na solusyon. Sa ganitong paraan, gumagalaw ang algorithm sa bawat iteration patungo sa mga solusyon na may mas mababang energy. Karaniwang sapat ang halos sampung iteration para mag-converge sa isang solusyon, at sa kabuuan ay nangangailangan ng mas kaunting iteration kaysa sa mga variational algorithm, na nasa hanay ng halos 100 iteration.

Pinagsasama ng optimizer ang bf-DCQO algorithm sa classical post-processing. Pagkatapos ng pagsukat ng distribusyon ng mga estado, nagsasagawa ng local search. Sa panahon ng local search, random na binibigo ang mga bit ng nasukatang solusyon. Pagkatapos ng flip, sinusuri ang energy ng bagong bitstring. Kung mas mababa ang energy, itinatago ang bitstring bilang bagong solusyon. Linearly lamang nag-scale ang local search sa bilang ng qubit; kaya naman, ito ay computationally mura. Dahil itinutuwid ng post-processing ang mga local na bitflip, binabayaran nito ang mga bit-flip error na madalas na resulta ng mga imperpeksyon ng hardware at mga readout error.

### Workflow
Sumusunod ang isang schematic ng workflow ng Quantum Optimizer.

![Workflow](../docs/images/guides/kipu-optimization/workflow.svg "Workflow ng Quantum Optimizer")

Sa pamamagitan ng paggamit ng Quantum Optimizer, mababawasan ang paglutas ng optimization problem sa quantum hardware sa:
* Bumugin ang objective function ng problema
* I-access ang Optimizer sa pamamagitan ng Qiskit Functions
* Patakbuhin ang Optimizer at kolektahin ang resulta

## Mga Benchmark
Ipinapakita ng mga benchmark metric sa ibaba na epektibong tinutugunan ng Optimizer ang mga problemang may hanggang 156 qubit at nagbibigay ng pangkalahatang-ideya ng katumpakan at scalability ng optimizer sa iba't ibang uri ng problema. Tandaan na ang aktwal na mga performance metric ay maaaring mag-iba depende sa mga katangian ng partikular na problema, tulad ng bilang ng variable, density at locality ng mga term sa objective function, at ang polynomial order.

Ang sumusunod na talahanayan ay kinabibilangan ng approximation ratio (AR), isang metric na tinukoy tulad ng sumusunod:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
kung saan ang $C$ ay ang objective function, $C_{\textrm{min}}$, $C_{\textrm{max}}$ ang mga minimum at maximum na halaga nito, at $C^{*}$ ang halaga ng pinakamahusay na solusyong nahanap, ayon sa pagkakasunod. Kaya naman, ang AR=100\% ay nangangahulugang nakuha na ang ground state ng problema.

| Halimbawa         | Bilang ng qubit | Approximation Ratio | Kabuuang oras (s) | Runtime usage (s) | Kabuuang bilang ng shot | Bilang ng iteration |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| Unweighted MaxCut |        28        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        30        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        32        |   100%   |      180       |        30         |          30k          |          5           |
| Unweighted MaxCut |        80        |   100%   |      480       |        60         |          90k          |          9           |
| Unweighted MaxCut |       100        |   100%   |      330       |        60         |          60k          |          6           |
| Unweighted MaxCut |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Ang mga MaxCut instance na may 28, 30, at 32 qubit ay pinatakbo sa ibm_sherbrooke. Ang mga instance na may 80, 100, at 120 ay pinatakbo sa isang Heron r2 processor.
- Ang mga HUBO instance ay pinatakbo rin sa isang Heron r2 processor.

Lahat ng benchmark instance ay available sa GitHub (tingnan ang [Kipu benchmark instances](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Makikita ang isang halimbawa ng pagpapatakbo ng mga instance na ito sa [Halimbawa 3: Mga benchmark instance](#example-3-benchmark-instances).

## Mga input at output
### Input
Tingnan ang sumusunod na talahanayan para sa lahat ng input parameter na tinatanggap ng Quantum Optimizer. Ang susunod na seksyon ng **Mga Opsyon** ay nagbibigay ng mas detalyadong impormasyon tungkol sa mga available na `options`.

| Pangalan     | Uri                | Paglalarawan                                                                                                                                                                            | Kinakailangan | Default                                                                                       | Halimbawa                                                                                                            |
| ------------ | ------------------ | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------- | --------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| problem      | `Dict[str, float]` | Ang mga coefficient ng optimization problem na nabuuo bilang QUBO/HUBO o spin format. Para sa karagdagang impormasyon tungkol sa detalye ng problema, tingnan ang [Mga tinatanggap na format ng problema](#accepted-problem-formats)                                       | Oo      | N/A                                                                                           | `{"()": -21.0, "(0, 4)": 0.5,"(0, 2)": 0.5,"(0, 1)": 0.5,"(1, 3)": 0.5}`                                             |
| problem_type | `str`              | Tukuyin kung ang mga coefficient ng problema ay nasa binary (QUBO/HUBO) o spin format. Ang dalawang posibilidad ay `"spin"` o `"binary"`                                                   | Oo      | N/A                                                                                           | `"spin"`                                                                                                             |
| backend_name | `str`              | Pangalan ng backend para gawin ang query                                                                                                                                                  | Oo       | N/A                                                      | `"ibm_fez"`                                                                                                          |                                                                                         |
| options      | `Dict[str, Any]`   | Mga opsyon para hawakan ang hardware submission, tulad ng bilang ng shot. Para sa karagdagang detalye sa configuration ng mga opsyon, tingnan ang seksyon ng [Mga Opsyon](#options) | Hindi       | Para makita ang mga default na halaga ng configuration ng mga opsyon tingnan ang seksyon ng [Mga Opsyon](#options)                                                         | `{"shots": 5000, "num_iterations": 3, "use_session": True, "seed_transpiler": 42}` |

#### Mga tinatanggap na format ng problema
Ang mga argumento ng `problem` at `problem_type` ay nag-encode ng optimization problem ng form na:

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
kung saan

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- Sa pamamagitan ng pagpili ng `problem_type = "binary"`, tinutukoy mo na ang cost function ay nasa `binary` format, na nangangahulugang $D = {0,  1}^{n}$, ibig sabihin, ang cost function ay nakasulat sa QUBO/HUBO formulation.
- Sa kabilang banda, sa pamamagitan ng pagpili ng `problem_type = "spin"`, ang cost function ay nakasulat sa Ising formulation, kung saan $D = {-1, 1}^{n}$.

Ang mga coefficient ng problema ay dapat na i-encode sa isang dictionary tulad ng sumusunod:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"} &: \quad &g_{k_1, ..., k_m}, \\ \nonumber
&\texttt{}}
\end{align}
$$

- Pakitandaan na ang mga key ng dictionary ay dapat na mga string na naglalaman ng valid na tuple ng mga non-repeating na integer.

#### Mga Opsyon
Nagbibigay ang Iskay ng mga kakayahan sa fine-tuning sa pamamagitan ng mga opsyonal na parameter. Habang ang mga default ay gumagana nang maayos para sa karamihan ng mga problema, maaari mong i-customize ang gawi para sa mga tiyak na kinakailangan:

| Parameter | Uri | Default | Paglalarawan |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | Quantum measurement bawat iteration (mas mataas = mas tumpak) |
| **num_iterations** | `int` | 10 | Mga iteration ng algorithm (ang mas maraming iteration ay maaaring mapabuti ang kalidad ng solusyon) |
| **use_session** | `bool` | True | Gumamit ng IBM session para sa mas mababang oras ng pila |
| **seed_transpiler** | `int` | None | Itakda para sa reproducible na quantum circuit compilation |
| **direct_qubit_mapping** | `bool` | False | I-map ang mga virtual qubit nang direkta sa mga physical qubit |
| **job_tags** | `List[str]` | None | Mga custom na tag para sa job tracking |
| **preprocessing_level** | `int` | 0 | Intensity ng problem preprocessing (0-3) - tingnan ang mga detalye sa ibaba |
| **postprocessing_level** | `int` | 2 | Antas ng refinement ng solusyon (0-2) - tingnan ang mga detalye sa ibaba |
| **transpilation_level** | `int` | 0 | Mga transpiler optimization trial (0-5) - tingnan ang mga detalye sa ibaba |
| **transpile_only** | `bool` | False | Suriin ang circuit optimization nang hindi pinapatakbo ang buong execution |

**Mga Antas ng Preprocessing (0-3)**: Espesyal na mahalaga para sa mga mas malalaking problema na hindi kasalukuyang akma sa coherence time ng hardware. Ang mas mataas na antas ng preprocessing ay nagkakamit ng mas mababang circuit depth sa pamamagitan ng mga approximation sa problem transpilation:
- **Antas 0**: Eksaktong, mas mahabang circuit
- **Antas 1**: Magandang balanse sa pagitan ng katumpakan at approximation, tinatanggal lamang ang mga gate na may mga anggulo sa pinakamababang ika-10 percentile
- **Antas 2**: Bahagyang mas mataas na approximation, tinatanggal ang mga gate na may mga anggulo sa pinakamababang ika-20 percentile at gumagamit ng `approximation_degree=0.95` sa transpilation
- **Antas 3**: Pinakamataas na antas ng approximation, tinatanggal ang mga gate sa pinakamababang ika-30 percentile at gumagamit ng `approximation_degree=0.90` sa transpilation

**Mga Antas ng Transpilation (0-5)**: Kontrolin ang mga advanced transpiler optimization trial para sa quantum circuit compilation. Maaaring magdulot ito ng pagtaas sa classical overhead, at para sa ilang kaso maaaring hindi ito magbago ng circuit depth. Ang default na halaga na `2` sa pangkalahatan ay nagdudulot ng pinakamaliit na circuit at medyo mabilis.
- **Antas 0**: Optimization ng decomposed DCQO circuit (layout, routing, scheduling)
- **Antas 1**: Optimization ng `PauliEvolutionGate` at pagkatapos ay ang decomposed DCQO circuit (max_trials=10)
- **Antas 2**: Optimization ng `PauliEvolutionGate` at pagkatapos ay ang decomposed DCQO circuit (max_trials=15)
- **Antas 3**: Optimization ng `PauliEvolutionGate` at pagkatapos ay ang decomposed DCQO circuit (max_trials=20)
- **Antas 4**: Optimization ng `PauliEvolutionGate` at pagkatapos ay ang decomposed DCQO circuit (max_trials=25)
- **Antas 5**: Optimization ng `PauliEvolutionGate` at pagkatapos ay ang decomposed DCQO circuit (max_trials=50)

**Mga Antas ng Postprocessing (0-2)**: Kontrolin kung gaano karaming classical optimization ang ginagawa, binabayaran ang mga bit-flip error gamit ang iba't ibang bilang ng greedy pass ng local search:
- **Antas 0**: 1 pass
- **Antas 1**: 2 pass
- **Antas 2**: 3 pass

**Transpile-only mode**: Available na ngayon para sa mga gumagamit na gustong suriin ang circuit optimization nang hindi pinapatakbo ang buong quantum algorithm execution.

**Halimbawa ng custom na configuration**: Narito kung paano mo maaaring i-configure ang Iskay gamit ang iba't ibang setting:

In [1]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Seed optimization**: Tandaan na ang `seed_transpiler` ay nakatakda sa `None` bilang default. Pinagana nito ang awtomatikong proseso ng optimization ng transpiler. Kapag `None`, magsisimula ang system ng isang trial na may maraming seed at pipiliin ang isa na nagdudulot ng pinakamainam na circuit depth, ginagamit ang buong kapangyarihan ng parameter na `max_trials` para sa bawat antas ng transpilation.

**Transpilation level performance**: Ang pagtaas ng bilang ng `max_trials` na may mas mataas na halaga para sa `transpilation_level` ay hindi maiwasang magpapataas ng oras ng transpilation, ngunit maaaring hindi laging magbago ang panghuling circuit — ito ay lubos na nakasalalay sa partikular na circuit structure at kumplikasyon. Para sa ilang circuit/problema, gayunpaman, ang pagkakaiba sa pagitan ng 10 trial (antas 1) at 50 trial (antas 5) ay maaaring malaki, kaya ang paggalugad ng mga parameter na ito ay maaaring maging susi sa matagumpay na paghanap ng solusyon.

### Output
| Pangalan   | Uri               | Paglalarawan                                                           | Halimbawa                                                                |
| ------ | ------------------ | ----------------------------------------------------------------------| ---------------------------------------------------------------------- |
| result | `Dict[str, Any]` | Solusyon at metadata. Ang istruktura ay nag-iiba batay sa opsyon ng `transpile_only`. |   Tingnan ang "Mga nilalaman ng result dictionary" sa ibaba      |

### Mga nilalaman ng result dictionary
Ang istruktura ng result dictionary ay nakasalalay sa execution mode:

| Field         | Uri               | Mode | Paglalarawan                                                                                                             | Halimbawa |
| ------------ | :------------------: | :--: | ----------------------------------------------------------------------------------------------------------------------- | ------- |
| **solution**     | `Dict[str, int]` | **Standard** | Ang sorted mapped solution kung saan ang mga key ay mga index ng variable (bilang mga string) na naayos nang numerically at ang mga value ay ang kaukulang halaga ng variable (1/-1 para sa mga spin problem, 1/0 para sa mga binary problem).| `{'0': -1, '1': -1, '2': -1, '3': 1, '4': 1}` |
| **solution_info**      | `Dict[str, Any]`   | **Standard** | Detalyadong impormasyon tungkol sa solusyon (tingnan ang mga detalye sa ibaba)                                           | `{'bitstring': '11100', 'cost': -13.8, 'seed_transpiler': 42, 'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}}` |
| **prob_type** | `str`             | **Standard** | Ang uri ng optimization problem ('spin' o 'binary')                                                           | `'spin'` |
| **transpilation_info** | `Dict[str, Any]` | **Transpile-only** | Pagsusuri ng circuit at mga detalye ng transpilation (tingnan ang mga detalye sa ibaba) | `{'best_seed': 42, 'transpilation_time_seconds': 50.06, 'transpiled_circuit': {'depth': 576, 'gate_count': 4177, 'num_qubits': 156, 'width': 176, 'operations': {'sx': 1325, 'rx': 891, 'cz': 783, 'rz': 650, 'rzz': 466, 'x': 42, 'measure': 20}}}` |

#### Standard execution
Kapag ang opsyonal na parameter na `transpile_only=False`:

**solution_info dictionary:**
- **"bitstring"** (`str`): Ang raw bitstring representation ng solusyon.
- **"cost"** (`float`): Ang halaga ng cost/energy na nauugnay sa solusyon.
- **"seed_transpiler"** (`int`): Ang random seed na ginamit para sa transpiler na nagdulot ng resultang ito.
- **"mapping"** (`Dict[int, int]`): Ang orihinal na qubit-to-variable mapping na ginamit sa computation.
- **"qpu_time"** (`float`, opsyonal): Ang oras ng QPU execution sa segundo.

**Mga tala tungkol sa variable mapping:**
- Ang dictionary ng `solution` ay nakuha mula sa solution bitstring, gamit ang object na `mapping` para sa pag-index ng mga variable.
- Kapag `problem_type=spin` ginagamit natin ang assignment na $1 \rightarrow -1, \quad 0 \rightarrow 1$.
- Ang mga key sa solution dictionary ay mga index ng variable na naayos nang numerically bilang mga string.

#### Transpilation analysis
Kapag ang opsyonal na parameter na `transpile_only=True`:

**transpilation_info dictionary:**
- **"best_seed"** (`int`): Ang pinakamainam na seed na nahanap para sa transpilation
- **"transpilation_time_seconds"** (`float`): Oras na kinuha para sa proseso ng transpilation
- **"transpiled_circuit"** (`Dict`): Pagsusuri ng circuit na naglalaman ng:
  - **"depth"** (`int`): Circuit depth (bilang ng layer)
  - **"gate_count"** (`int`): Kabuuang bilang ng gate sa circuit
  - **"num_qubits"** (`int`): Bilang ng qubit na ginamit
  - **"width"** (`int`): Lapad ng circuit
  - **"operations"** (`Dict[str, int]`): Bilang ng bawat uri ng gate na ginamit

**Paggamit ng transpile-only mode:**
- Available para sa mga gumagamit na gustong suriin ang circuit optimization nang hindi pinapatakbo ang buong quantum algorithm execution.
- Kapaki-pakinabang para sa pagsusuri ng circuit, pag-aaral ng depth optimization, at pag-unawa sa mga epekto ng transpilation bago sumailalim sa buong execution.

## Magsimula
Sa dokumentasyong ito, dadaan tayo sa mga hakbang ng paggamit ng Iskay Quantum Optimizer. Sa proseso ay mabilis naming ipapakita kung paano i-load ang function mula sa catalog at kung paano i-convert ang iyong problema sa valid na input, habang ipinapakita kung paano ka mag-eksperimento sa iba't ibang opsyonal na parameter.

Para sa mas detalyadong halimbawa, pakitingnan ang tutorial na [Solve the Market Split problem with Kipu Quantum's Iskay Quantum Optimizer](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), kung saan dinadaan tayo sa buong proseso ng paggamit ng Iskay Solver para tugunan ang Market Split problem, na kumakatawan sa isang real-world na hamon sa resource allocation kung saan ang mga merkado ay kailangang hatiin sa mga balanseng rehiyon ng benta para matugunan ang eksaktong mga target na demand.

Mag-authenticate gamit ang iyong API key, na makikita sa [IBM Quantum Platform dashboard](http://quantum.cloud.ibm.com/), at piliin ang Qiskit Function tulad ng sumusunod:

In [ ]:
# ruff: noqa: F821

> **Note:** Ipinapalagay ng sumusunod na code na nai-save mo na ang iyong mga kredensyal. Kung hindi pa, sundin ang mga tagubilin sa [i-save ang iyong IBM Cloud account](/guides/functions#install-qiskit-functions-catalog-client) para mag-authenticate gamit ang iyong API key.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Halimbawa 1: Simpleng cost function

Isaalang-alang ang cost function sa spin formulation:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

kung saan $(x_0, ..., x_4) \in {-1, 1}^5$.

Ang solusyon sa simpleng cost function na ito ay
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
na may minimum na halaga na $C^{*} = -6$

### 1. Likhain ang objective function

Nagsisimula tayo sa paglikha ng dictionary na may mga coefficient ng objective function tulad ng sumusunod:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Patakbuhin ang Optimizer
Nireresolba natin ang problema sa pamamagitan ng pagpapatakbo ng optimizer. Dahil $(x_0, ..., x_4) \in {-1, 1}^5$ kailangan nating itakda ang `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Kunin ang resulta
Ang solusyon ng optimization problem ay direktang ibinibigay mula sa optimizer.

In [ ]:
print(job.result())

Ipapakita nito ang isang dictionary ng form na:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Pansinin na ipinapakita ng dictionary na `solution` ang result vector $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Halimbawa 2: MaxCut

Maraming graph problem tulad ng MaxCut o Maximum independent set ang mga NP-hard na problema at mga ideal na kandidato para sa pagsubok ng mga quantum algorithm at hardware. Ipinapakita ng halimbawang ito ang paglutas ng MaxCut problem ng isang 3-regular graph gamit ang Quantum Optimizer.

Para patakbuhin ang halimbawang ito kailangan mong i-install ang `networkx` package bukod sa `qiskit-ibm-catalog`. Para i-install ito, patakbuhin ang sumusunod na command:

In [ ]:
# %pip install networkx numpy

### 1. Likhain ang objective function
Magsimula sa paglikha ng random 3-regular graph. Para sa graph na ito, tinutukoy natin ang objective function ng MaxCut problem.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the Max-Cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Patakbuhin ang Optimizer
Lutasin ang problema sa pamamagitan ng pagpapatakbo ng optimizer.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Kunin ang resulta
Kunin ang resulta at i-map ang solution bitstring pabalik sa mga orihinal na node ng graph.

In [ ]:
print(job.result())

Ang solusyon sa Maxcut problem ay direktang nilalaman sa sub-dictionary na `solution` ng result object

In [ ]:
maxcut_solution = job.result()["solution"]

## Halimbawa 3: Mga benchmark instance
Ang mga benchmark instance ay available sa GitHub: [Kipu benchmark instances.](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)

Maaaring i-load ang mga instance gamit ang `pygithub` library. Para i-install ito, patakbuhin ang sumusunod na command:

In [ ]:
# %pip install pygithub

Ang mga path para sa mga benchmark instance ay:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Para maisagawa muli ang performance ng benchmark para sa mga HUBO instance, piliin ang backend na `ibm_marrakesh` at itakda ang `direct_qubit_mapping` sa `True` sa sub-dictionary ng `options`.
Ang sumusunod na halimbawa ay nagpapatakbo ng Maxcut instance na may 32 node.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_32_nodes.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "ibm_brisbane",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Mga use case
Ang mga tipikal na use case para sa Optimization solver ay mga combinatorial optimization problem. Maaari kang magresolba ng mga problema mula sa maraming industriya tulad ng pananalapi, pharmaceutics, o logistics. Narito ang ilang halimbawa.
* Portfolio optimization (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.054037) at [white paper](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Protein folding (HUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Logistics scheduling (QUBO): [scientific publication](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Network optimization: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Market split (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Kung interesado kang harapin ang isang tiyak na use case at bumuo ng dedicated na mapping, maaari kaming tumulong. [Makipag-ugnayan sa amin.](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)

## Makakuha ng suporta
Para sa suporta, makipag-ugnayan sa support@kipu-quantum.com.

## Mga susunod na hakbang
- [Humiling ng access sa Quantum Optimizer ng Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5)
- Subukan ang tutorial na [Solve the Market Split problem with Kipu Quantum's Iskay Quantum Optimizer](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Suriin ang [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477.](https://arxiv.org/abs/2409.04477)
- Suriin ang [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898.](https://arxiv.org/abs/2405.13898)
- Suriin ang [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663.](https://arxiv.org/abs/2505.08663)

## Karagdagang impormasyon
Ang Iskay, tulad ng pangalan ng aming kumpanya na Kipu Quantum, ay isang salitang Peruvian. Bagaman kami ay isang startup mula sa Germany, ang mga salitang ito ay nagmumula sa katutubong bansa ng isa sa aming mga co-founder, kung saan ang Quipu ay isa sa mga unang makina ng pagkalkula na binuo ng sangkatauhan 2000 taon BCE.